# ScaleRAG – Multimodal RAG Retrieval Pipeline (v1)

This notebook builds the first version of our **multimodal Retrieval-Augmented Generation (RAG)** pipeline using the preprocessed chunks from the data preparation stage. We load the merged text and image chunks and compute dense embeddings for retrieval.

A **SentenceTransformer model** (`all-MiniLM-L6-v2`) produces **384-dimensional embeddings** for text, while **OpenAI’s CLIP model** (`ViT-B/32`) produces **512-dimensional embeddings** for images.  
When a chunk has both an image and caption, we concatenate the normalized caption and image vectors to form an **896-dimensional embedding**.  
All embeddings are normalized and saved, and **FAISS indexes** are built for fast similarity search.  

---

## Pipeline Overview

1. **Model Setup:**  
   Initialize the SentenceTransformer for text and the CLIP model (with processor) for images, and configure the compute device (GPU/CPU).

2. **Load Chunks:**  
   Read all preprocessed RAG chunk files from `data/rag_chunks/*.json` into a single list.

3. **Separate by Type:**  
   Split the merged chunks into text chunks (*paragraph, text, equation*) and image chunks (*figures, tables*).

4. **Embed Text Chunks:**  
   Batch-encode all text contents using the text model, normalize the 384-D vectors, and attach them to chunk records.

5. **Embed Image Chunks:**  
   For each figure/table, use CLIP to extract a 512-D image feature, normalize it, and if a caption exists, encode it with the text model and concatenate (resulting in 896-D). Attach the combined embeddings.

6. **Save Embeddings:**  
   Store the list of embedded chunks to disk in both JSON and Pickle formats for reuse.

7. **Build FAISS Indexes:**  
   Collect text and image vectors into NumPy arrays, create **FAISS IndexFlatIP** indexes (inner-product for cosine similarity), add the vectors, and save the indexes to disk.

8. **Retrieval Demo:**  
   Define query functions to embed user queries and retrieve the top-k relevant text or image chunks via the FAISS indexes.

---

## Output Directory Summary

- `data/RAG/embeddings/all_papers.embeddings.json` – JSON file of all chunk records with embeddings  
- `data/RAG/embeddings/all_papers.embeddings.pkl` – Pickle file of the same embedding data  
- `data/RAG/indexes/text.index.faiss` – FAISS index for text embeddings (384-D)  
- `data/RAG/indexes/image.index.faiss` – FAISS index for image embeddings (896-D)



## Model and Tokenizer Setup (CLIP & SentenceTransformer)

- **SentenceTransformer:**  
  Uses `all-MiniLM-L6-v2` for 384-dimensional text embeddings.

- **CLIP Model:**  
  Loads `openai/clip-vit-base-patch32` and its processor for 512-dimensional image feature extraction.

- **Device Configuration:**  
  Sets `device = cuda` if available, otherwise `cpu`. The CLIP model is moved to this device.  
  The text model remains on CPU in this notebook (but can also be moved to GPU if desired).

In [5]:
import torch
from PIL import Image
# Text model (SentenceTransformer)
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Vision model (CLIP)
from transformers import CLIPProcessor, CLIPModel
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

# Ensure models are on CPU or GPU as available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = clip_model.to(device)
text_model = text_model


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## Loading Preprocessed RAG Chunks

In this step, we load all preprocessed chunk files created during the data preparation phase.

1. **Load JSON Files:**  
   Read all chunk files from `data/rag_chunks/*.json`.  
   Each file contains a list of chunk records with fields such as `id`, `type`, `content`, and `metadata`.

2. **Merge Lists:**  
   Combine the contents of all files into a single list named `merged_chunks`.  
   Print the total number of chunks and display one example record to confirm the format.


In [ ]:
import json
import glob

merged_chunks = []

for file_path in glob.glob("data/rag_chunks/*.json"):
    with open(file_path, "r") as f:
        file_chunks = json.load(f)
        merged_chunks.extend(file_chunks)

print("Total merged chunks:", len(merged_chunks))
print("First example chunk:")
print(merged_chunks[0])


## Separating Text and Image Chunks

In this step, we categorize the merged RAG chunks based on their content type.

1. **Filter by Type:**  
   Create two separate lists from `merged_chunks`:  
   - `text_chunks` → where `type` is `"paragraph"`, `"text"`, or `"equation"`  
   - `image_chunks` → where `type` is `"figure"` or `"table"`

2. **Sanity Check:**  
   Print the counts of text vs. image chunks to verify the split.  
   This ensures that each chunk type will later be embedded using the correct model (text or image encoder).


In [ ]:
import numpy as np
from torch.nn.functional import normalize

embedded_chunks = []  # final list of chunk dicts with embeddings

text_chunks = [ch for ch in merged_chunks if ch['type'] in ['paragraph', 'text', 'equation']]
image_chunks = [ch for ch in merged_chunks if ch['type'] in ['figure', 'table']]

print(f"📄 Text chunks: {len(text_chunks)} | 🖼️ Image chunks: {len(image_chunks)}")

In [ ]:
len(text_chunks), len(image_chunks)

In [ ]:
text_chunks[0]

## Embedding Text Chunks

In this step, we compute dense embeddings for all textual RAG chunks using the SentenceTransformer model.

1. **Batch Encoding:**  
   Extract all text contents from `text_chunks` and encode them in batches using:  
   ```python
   text_model.encode(texts, batch_size=32)
   ```  
   The output is a NumPy array of shape *(num_text_chunks, 384)*, where each row represents a 384-dimensional embedding vector.

2. **Normalization:**  
   Normalize each 384-D vector by dividing it by its L2 norm so that all vectors lie on the unit sphere.  
   This ensures that cosine similarity can be computed directly using the inner product.

3. **Attach to Chunks:**  
   For each text chunk, add the normalized embedding (converted to a Python list) under the key `"embedding"`.  
   The updated record structure becomes:  
    ```python
   {
       "id": ...,
       "type": ...,
       "content": ...,
       "metadata": ...,
       "embedding": [...]
   }
   ```

In [ ]:
# Batch compute text embeddings for all text chunks to speed up
texts = [ch['content'] for ch in text_chunks]
text_embeds = text_model.encode(texts, batch_size=32, show_progress_bar=True)
# text_embeds will be a numpy array of shape (len(text_chunks), 384) in this case

# Normalize text embeddings
text_embeds = text_embeds / np.linalg.norm(text_embeds, axis=1, keepdims=True)

# Assign back the text embeddings to their chunks
for ch, vec in zip(text_chunks, text_embeds):
    vec_list = vec.tolist()
    ch_emb = vec_list  # 384-dim text embedding
    # we handle that in the image loop instead.
    embedded_chunks.append({
        "id": ch["id"],
        "type": ch["type"],
        "content": ch["content"],
        "metadata": ch.get("metadata", {}).copy(),
        "embedding": ch_emb
    })

print(f"Text embeddings computed for {len(text_chunks)} chunks.")

## Embedding Image Chunks (Figures/Tables)

In this step, we generate embeddings for all image-based RAG chunks such as figures and tables.

1. **Open Image:**  
   For each chunk in `image_chunks`, open the associated image file using PIL:  
   ```python
   image = Image.open(path).convert("RGB")
   ```  
   Skip any chunks where the image cannot be loaded or the file path is missing.

2. **CLIP Encoding:**  
   Preprocess each image using the CLIP processor and extract image features.
   This produces a **512-dimensional image feature vector**.

3. **Normalize Image Vector:**  
   Divide the 512-D image vector by its L2 norm to ensure unit length.

4. **Caption Encoding (if present):**  
   If the chunk’s content contains a caption text, encode it using the text model to obtain a **384-D vector**, then normalize it.

5. **Combine Features:**  
   - If the two vectors have matching shapes (rare case), average them.  
   - Otherwise, concatenate the 384-D caption vector and 512-D image vector to form an **896-D combined embedding**.

6. **Attach to Chunks:**  
   Add the combined 896-D embedding (converted to a Python list) under the key `"embedding"`.  
   The final record structure becomes:  
   ```python
   {
       "id": ...,
       "type": ...,
       "content": ...,
       "metadata": ...,
       "embedding": [...]
   }
   ```


In [ ]:
# Now handle image-containing chunks (figures, tables)
for ch in image_chunks:
    img_path = ch.get("metadata", {}).get("image_path") or ch.get("image_path")
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Warning: could not open image at {img_path}: {e}")
        continue  # skip if image not available
    # Preprocess image for CLIP
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    # Get CLIP image feature (512-dim)
    with torch.no_grad():
        image_feat = clip_model.get_image_features(pixel_values=pixel_values)
    image_vec = image_feat.cpu().numpy().flatten()
    # Normalize image embedding
    image_vec = image_vec / np.linalg.norm(image_vec)

    # Check if there's associated text (e.g., a caption in content)
    combined_vec = image_vec
    if ch.get("content"):
        caption = str(ch["content"])
        # Use the same text model for caption
        caption_emb = text_model.encode([caption])[0]
        caption_emb = caption_emb / np.linalg.norm(caption_emb)
        # Concatenate or average with image_vec:
        if caption_emb.shape[0] == image_vec.shape[0]:
            # If by chance using CLIP text encoder for caption, shapes align (512 each)
            combined_vec = (image_vec + caption_emb) / 2.0  # average
        else:
            # Different dimensions (e.g., 384 vs 512), concatenate
            combined_vec = np.concatenate([caption_emb, image_vec])
            # (combined_vec is now 896-dim in this scenario)
            # We could optionally reduce dimension or keep as is for indexing.
    else:
        # No caption text, combined_vec stays as image_vec
        pass

    embedded_chunks.append({
        "id": ch["id"],
        "type": ch["type"],
        "content": ch.get("content", ""),  # might be caption or empty
        "metadata": ch.get("metadata", {}).copy(),
        "embedding": combined_vec.tolist()
    })

print(f"Computed embeddings for {len(embedded_chunks)} chunks.")
# Show an example of a text chunk and an image chunk embedding (truncated for display)
for ex in embedded_chunks[:2]:
    print(f"{ex['type']} chunk '{ex['id']}' -> embedding length {len(ex['embedding'])}, sample: {ex['embedding'][:5]}")

## Saving Embeddings to Disk

In this step, we save the computed embeddings for future use.

1. **Create Directory:**  
   Ensure the folder `data/RAG/embeddings` exists.

2. **Save JSON:**  
   Dump the `embedded_chunks` list to `all_papers.embeddings.json`.

3. **Save Pickle:**  
   Write the same list to `all_papers.embeddings.pkl` using `pickle.dump`.

4. **Verification:**  
   Print confirmation messages after saving to confirm that both files were successfully created.


In [ ]:
import os, json, pickle

# Create the folder hierarchy
base_dir = "data/RAG"
emb_dir = os.path.join(base_dir, "embeddings")
os.makedirs(emb_dir, exist_ok=True)

print(f" +++ Folder created at: {emb_dir}")

# Save embeddings
json_path = os.path.join(emb_dir, "all_papers.embeddings.json")
pkl_path = os.path.join(emb_dir, "all_papers.embeddings.pkl")

# Save JSON
with open(json_path, "w") as f:
    json.dump(embedded_chunks, f)
print(f" +++ JSON embeddings saved to {json_path}")

# Save Pickle
with open(pkl_path, "wb") as f:
    pickle.dump(embedded_chunks, f)
print(f" +++ Pickle embeddings saved to {pkl_path}")


## Loading and Verifying Saved Embeddings

In this step, we perform a quick sanity check to ensure that the saved embedding files are valid and correctly structured.

1. **Load Pickle:**  
   Reload the embeddings list from `all_papers.embeddings.pkl`.

2. **Inspect Structure:**  
   Print the total number of chunks, view example keys from the first record (`id`, `type`, `content`, etc.), and check the embedding dimensions.

3. **Verification:**  
   Confirm that text chunks have **384-D embeddings** and image chunks have **896-D embeddings**.


In [6]:
import pickle

with open("data/RAG/embeddings/all_papers.embeddings.pkl", "rb") as f:
    embedded_chunks = pickle.load(f)

print("Loaded", len(embedded_chunks), "chunks.")
print("Example record keys:", embedded_chunks[0].keys())
print("Example metadata:", embedded_chunks[0]["metadata"])
print("Embedding dim:", len(embedded_chunks[0]["embedding"]))

Loaded 5447 chunks.
Example record keys: dict_keys(['id', 'type', 'content', 'metadata', 'embedding'])
Example metadata: {'page': 1, 'section': 'Abstract', 'bbox': [54.0, 415.423, 295.771, 519.616]}
Embedding dim: 384


In [7]:
import numpy as np

text_dims = [len(ch["embedding"]) for ch in embedded_chunks if ch["type"] in ["text","paragraph","equation"]]
image_dims = [len(ch["embedding"]) for ch in embedded_chunks if ch["type"] in ["figure","table"]]

print(f"Text embeddings: mean {np.mean(text_dims):.0f} ± {np.std(text_dims):.1f}")
print(f"Image embeddings: mean {np.mean(image_dims):.0f} ± {np.std(image_dims):.1f}")

Text embeddings: mean 384 ± 0.0
Image embeddings: mean 896 ± 0.0


## Splitting Embeddings for Indexing

In this step, we separate the embeddings by type to prepare for FAISS indexing.

1. **Filter by Dimension:**  
   Iterate over all loaded `embedded_chunks`.  
   - If the type is a text type and the embedding length is **384**, append it to `text_records` and collect its vector.  
   - If the type is an image/table and the embedding length is **896**, append it to `image_records` and collect its vector.  
   - Ignore any records with unexpected embedding dimensions.

2. **Stack Vectors:**  
   Convert the collected vectors into NumPy arrays:  
   - `text_vectors` → shape `(N_text, 384)`  
   - `image_vectors` → shape `(N_image, 896)`

3. **Print Shapes:**  
   Display the number of records and array shapes to verify the data integrity before indexing.


## Building and Saving FAISS Indexes

1. **Create Indexes:**  
   Initialize FAISS indexes using **inner product (IP)** similarity, which corresponds to cosine similarity when embeddings are normalized.  
   - `index_text = IndexFlatIP(384)` for text embeddings  
   - `index_image = IndexFlatIP(896)` for image embeddings

2. **Add Vectors:**  
   Add `text_vectors` to `index_text` and `image_vectors` to `index_image`.  
   The final index sizes should match the number of added vectors.

3. **Save to Disk:**  
   Ensure the folder `data/RAG/indexes` exists.  
   Save the indexes as `text.index.faiss` and `image.index.faiss`, and print confirmation messages to verify successful writes.


In [8]:
import numpy
import faiss
print("NumPy version:", numpy.__version__)
print("FAISS version:", faiss.__version__)

NumPy version: 1.26.4
FAISS version: 1.7.2


In [9]:
import numpy as np
import faiss

# --- Split into text chunks (384-D) and figure/table chunks (896-D)

text_records = []
text_vectors = []

image_records = []
image_vectors = []

for ch in embedded_chunks:
    vec = np.array(ch["embedding"], dtype="float32")
    dim = vec.shape[0]

    if ch["type"] in ["text", "paragraph", "equation", "table_caption", "figure_caption"]:
        # safety check: only keep consistent dim=384
        if dim == 384:
            text_records.append(ch)
            text_vectors.append(vec)
    elif ch["type"] in ["figure", "table"]:
        # safety check: only keep consistent dim=896 (caption+image concat case)
        if dim == 896:
            image_records.append(ch)
            image_vectors.append(vec)
    else:
        # ignore other types for now or print to debug
        pass

text_vectors = np.stack(text_vectors, axis=0) if len(text_vectors) > 0 else np.zeros((0,384), dtype="float32")
image_vectors = np.stack(image_vectors, axis=0) if len(image_vectors) > 0 else np.zeros((0,896), dtype="float32")

print("Text records:", len(text_records), "| text_vectors shape:", text_vectors.shape)
print("Image records:", len(image_records), "| image_vectors shape:", image_vectors.shape)

# --- Build FAISS indexes

dim_text = text_vectors.shape[1]
dim_image = image_vectors.shape[1]

index_text = faiss.IndexFlatIP(dim_text)    # cosine sim if vectors are normalized
index_image = faiss.IndexFlatIP(dim_image)  # cosine sim if vectors are normalized

index_text.add(text_vectors)
index_image.add(image_vectors)

print("FAISS index_text size:", index_text.ntotal)
print("FAISS index_image size:", index_image.ntotal)

# Save indexes to disk so you don't have to rebuild next time
import os
os.makedirs("data/RAG/indexes", exist_ok=True)

faiss.write_index(index_text, "data/RAG/indexes/text.index.faiss")
faiss.write_index(index_image, "data/RAG/indexes/image.index.faiss")

print(" +++ Saved FAISS indexes.")


Text records: 4707 | text_vectors shape: (4707, 384)
Image records: 740 | image_vectors shape: (740, 896)
FAISS index_text size: 4707
FAISS index_image size: 740
✅ Saved FAISS indexes.


## Defining Query and Retrieval Functions

In this section, we define helper functions that enable natural language queries over both text and image embeddings.

1. **Text Query Embedding:**  
   Define `embed_query_text(query)` to encode an input string using the text model and normalize it into a **384-D vector**.  
   This vector will be used for querying the text FAISS index.

2. **Image-side Query Embedding:**  
   Define `embed_query_clip_text(query)` to compute CLIP’s text embedding (**512-D**) and normalize it.  
   Then implement `build_query_for_image_index(query)` to concatenate:  
   - The normalized **text-model embedding (384-D)**  
   - The normalized **CLIP text embedding (512-D)**  
   forming a combined **896-D query vector** for the image index.

3. **Retrieve Functions:**  
   - `retrieve_text(query, k)`: Embeds the query, searches `index_text`, and returns the top-k text chunks with their scores and metadata.  
   - `retrieve_image(query, k)`: Builds the 896-D query vector, searches `index_image`, and returns the top-k figure or table chunks.

These retrieval functions enable semantic search over both textual and visual chunks using natural language queries.


In [13]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = clip_model.to(device)

def embed_query_text(query: str):
    """
    Embed the query using the SAME SentenceTransformer you used for text chunks.
    Returns a normalized 384-D np.array(float32) of shape (1, 384).
    """
    q_vec = text_model.encode([query])  # shape (1,384) as float32/float64
    q_vec = q_vec / np.linalg.norm(q_vec, axis=1, keepdims=True)
    return q_vec.astype("float32")

def embed_query_clip_text(query: str):
    """
    Use CLIP's text tower to embed the query in CLIP space (512-D).
    This lets us search the 'image side' because CLIP aligns text<->image.
    We'll L2-normalize it.
    """
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        # get_text_features gives CLIP's text embedding
        text_feat = clip_model.get_text_features(**inputs)
        # normalize
        text_feat = text_feat / text_feat.norm(p=2, dim=-1, keepdim=True)

    vec = text_feat.squeeze(0).detach().cpu().numpy()  # (512,)
    return vec.astype("float32")

def build_query_for_image_index(query: str):
    """
    Build the 896-D query vector that matches how figure/table embeddings were built:
    [384-D text_model embedding | 512-D CLIP text embedding]
    Both halves are normalized separately first.
    Returns shape (1, 896).
    """
    q_caption_384 = embed_query_text(query)[0]        # (384,)
    q_cliptext_512 = embed_query_clip_text(query)     # (512,)

    q_concat_896 = np.concatenate([q_caption_384, q_cliptext_512], axis=0)
    # final normalize whole vector so cosine/IP works nicely
    q_concat_896 = q_concat_896 / np.linalg.norm(q_concat_896, keepdims=True)

    return q_concat_896.astype("float32")[None, :]    # shape (1,896)


In [14]:
def retrieve_text(query, k=5):
    """
    Search the text FAISS index.
    Returns the top-k text chunks with scores and metadata.
    """
    q_vec = embed_query_text(query)  # (1,384)
    scores, idxs = index_text.search(q_vec, k)  # cosine/IP scores

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0])):
        ch = text_records[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "id": ch["id"],
            "type": ch["type"],
            "page": ch["metadata"].get("page"),
            "section": ch["metadata"].get("section"),
            "content_preview": ch["content"][:400],
        })
    return results

def retrieve_image(query, k=5):
    """
    Search the figure/table FAISS index using the 896-D query vector.
    Returns top-k visual chunks (figures/tables).
    """
    q_vec = build_query_for_image_index(query)  # (1,896)
    scores, idxs = index_image.search(q_vec, k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0])):
        ch = image_records[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "id": ch["id"],
            "type": ch["type"],
            "page": ch["metadata"].get("page"),
            "section": ch["metadata"].get("section"),
            "image_path": ch["metadata"].get("image_path"),
            "content_preview": ch.get("content", "")[:400],  # caption or ""
        })
    return results

def retrieve_multimodal(query, k_text=5, k_image=5):
    """
    Convenience wrapper: gets both text and image/table hits.
    """
    text_hits = retrieve_text(query, k=k_text)
    image_hits = retrieve_image(query, k=k_image)
    return text_hits, image_hits


## Retrieval Demo – Multimodal Query

In this step, we test the multimodal retrieval pipeline using a natural language question.  
The query is embedded for both text and image spaces, and the top-k most relevant chunks are retrieved from the FAISS indexes.

The output below displays:
- **Text hits:** top-5 paragraphs or sections ranked by similarity score.  
- **Image/Figure/Table hits:** top-5 visual elements with captions and page references.

This confirms that the RAG system can retrieve semantically relevant text and images for complex research-style queries.


In [15]:
question = "Tell me about the 'LLM in a Flash' paper. What is the novelty? what methods they proposed and what inference speed up they got?"
text_hits, image_hits = retrieve_multimodal(question, k_text=5, k_image=5)

print("=== TEXT HITS ===")
for h in text_hits:
    print(f"[rank {h['rank']} score {h['score']:.3f}] page {h['page']} section {h['section']}")
    print(h["content_preview"])
    print("---")

print("\n=== IMAGE / FIGURE / TABLE HITS ===")
for h in image_hits:
    print(f"[rank {h['rank']} score {h['score']:.3f}] {h['type']} page {h['page']} section {h['section']}")
    print("image_path:", h["image_path"])
    print("caption/summary:", h["content_preview"])
    print("---")


=== TEXT HITS ===
[rank 0 score 0.645] page 19 section C.5 Llama 2
Latency analysis. LLM in flash gets 3x speed up over naive baseline (Table 3). It is also performing better than hybrid model which is the theoretical lower bound for approaches that doesn't use sparsity.
---
[rank 1 score 0.598] page 14 section A Appendix Overview
In Appendix F, we go over implications of llm in flash when going to smaller devices.
---
[rank 2 score 0.538] page 1 section Introduction
However, the unprecedented capabilities of these models come with substantial computational and memory requirements for inference. LLMs can contain hundreds of billions or even trillions of parameters, which makes them challenging to load and run efficiently, especially on personal devices.
---
[rank 3 score 0.522] page 18 section C.4 Phi-2
We have applied LLM in Flash for Phi-2 models. We first relufied the model then trained the predictor and applied inference. Since the model is already small, we gave it 65% of its memo

## Building Combined Context for RAG Generation

Here we define the `build_context()` function, which merges top-ranked text and image retrievals into a unified context block.  
For each query, it gathers the top-k text and image hits, formats their content, and concatenates them into a single context string.  

This context will later be passed to the generation model to enable multimodal, context-aware responses.


In [16]:
def build_context(query, k_text=5, k_image=3):
    text_hits, image_hits = retrieve_multimodal(query, k_text, k_image)

    context_parts = []
    for h in text_hits:
        context_parts.append(f"[Text: page {h['page']} - {h['section']}]\n{h['content_preview']}")

    for h in image_hits:
        context_parts.append(
            f"[{h['type'].upper()}: page {h['page']} - {h['section']}]\n"
            f"Caption: {h['content_preview']}\nImage path: {h['image_path']}"
        )

    context = "\n\n".join(context_parts)
    return context


## Checking GPU Memory and Selecting Model Configuration

Before loading any large language models, we first check the available GPU memory to decide which model variant and precision can be used efficiently.

This step automatically selects an appropriate **Gemma model (2B or 7B)** and determines whether **4-bit quantization** should be applied based on available VRAM.  
It ensures optimal performance while preventing out-of-memory errors during model loading and inference.


In [14]:
# Get GPU available memory
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2**30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 15 GB


In [15]:
# Note: the following is Gemma focused, however, there are more and more LLMs of the 2B and 7B size appearing for local use.
if gpu_memory_gb < 5.1:
    print(f"Your available GPU memory is {gpu_memory_gb}GB, you may not have enough memory to run a Gemma LLM locally without quantization.")
elif gpu_memory_gb < 8.1:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in 4-bit precision.")
    use_quantization_config = True
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.")
    use_quantization_config = False
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb > 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommend model: Gemma 7B in 4-bit or float16 precision.")
    use_quantization_config = False
    model_id = "google/gemma-7b-it"

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

GPU memory: 15 | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.
use_quantization_config set to: False
model_id set to: google/gemma-2b-it


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available
from transformers import BitsAndBytesConfig
import time

# Quantization config
# ----------------------------
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)


# Attention optimization
# ----------------------------
if is_flash_attn_2_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")


# Load local model path
# ----------------------------
model_id = "./gemma_2b" 
use_quantization_config = True

print(f"[INFO] Using local model_id: {model_id}")

# ----------------------------
# Load tokenizer and model
# ----------------------------
start = time.time()

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=quantization_config if use_quantization_config else None,
    low_cpu_mem_usage=False,
    attn_implementation=attn_implementation,
    trust_remote_code=True
)

if not use_quantization_config:
    llm_model.to("cuda")

end = time.time()
print(f"Model loaded successfully in {end - start:.2f} seconds")


[INFO] Using attention implementation: sdpa
[INFO] Using local model_id: ./gemma_2b


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully in 32.96 seconds


In [17]:
# Test the model
inputs = tokenizer("Hello! How are you today?", return_tensors="pt").to("cuda")
outputs = llm_model.generate(**inputs, max_new_tokens=350,
    temperature=0.9,   # higher = more exploratory
    top_p=0.9)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Hello! How are you today?

I'm doing well, thank you for asking. I'm enjoying the beautiful weather today. How about you?

I'm doing well, thank you for asking! It's a beautiful day, perfect for enjoying the outdoors. I'm enjoying the sunshine and the fresh air.

I'm happy to hear that you're doing well as well! Is there anything I can do to help you today?


In [18]:
question = "What methods exist to accelerate token generation during inference without retraining the language model?"
# Build the RAG context from retrieved passages
context = build_context(question, k_text=10, k_image=5)

# Compose the final prompt
prompt = f"""
You are an expert research assistant specializing in AI and Machine Learning papers.
Your task is to answer questions clearly, factually, and concisely using only the provided retrieved context.

Guidelines:
- Base your answer only on the given context. Do not add external knowledge.
- If the context contains multiple relevant parts (text, tables, or figures), integrate them logically.
- If the context does not include the answer, say "Not mentioned in the retrieved context."
- Keep explanations precise and academic (1–2 short paragraphs maximum).

Question: {question}

Context:
{context}

Answer:
"""


In [19]:

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.9,
    top_p=0.9,
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("🧠 RAG Answer:\n")
print(answer)


🧠 RAG Answer:


You are an expert research assistant specializing in AI and Machine Learning papers.
Your task is to answer questions clearly, factually, and concisely using only the provided retrieved context.

Guidelines:
- Base your answer only on the given context. Do not add external knowledge.
- If the context contains multiple relevant parts (text, tables, or figures), integrate them logically.
- If the context does not include the answer, say "Not mentioned in the retrieved context."
- Keep explanations precise and academic (1–2 short paragraphs maximum).

Question: What methods exist to accelerate token generation during inference without retraining the language model?

Context:
[Text: page 10 - REFERENCES]
Sebastian Borgeaud, Arthur Mensch, Jordan Hoffmann, Trevor Cai, Eliza Rutherford, Katie Millican, George Bm Van Den Driessche, Jean-Baptiste Lespiau, Bogdan Damoc, Aidan Clark, et al. Improving language models by retrieving from trillions of tokens. In International confere

# Load LLaMA Model 

In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_flash_attn_2_available

# --- Model choice ---
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
use_quantization_config = True

# --- Quantization (for T4) ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# --- Flash Attention or SDPA ---
if is_flash_attn_2_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")

# --- Load tokenizer & model ---
start = time.time()
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=quantization_config if use_quantization_config else None,
    attn_implementation=attn_implementation,
    low_cpu_mem_usage=False,
    trust_remote_code=True
)

if not use_quantization_config:
    llm_model.to("cuda")

print(f"✅ Model {model_id} loaded in {time.time()-start:.1f}s")


In [24]:
question = "Tell me about the 'LLM in a Flash' paper. What is the novelty? what methods they proposed and what inference speed up they got?"

# --- Build the RAG context from retrieved passages ---
context = build_context(question, k_text=10, k_image=5)

# --- Compose the final prompt ---
prompt = f"""
You are an expert research assistant specializing in AI and Machine Learning papers.
Your task is to answer questions clearly, factually, and concisely using only the provided retrieved context.

Guidelines:
- Base your answer only on the given context. Do not add external knowledge.
- If the context contains multiple relevant parts (text, tables, or figures), integrate them logically.
- If the context does not include the answer, say "Not mentioned in the retrieved context."
- Keep explanations precise and academic (1–2 short paragraphs maximum).

Question: {question}

Context:
{context}

Answer:
"""


In [25]:
# Tokenize the composed prompt
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate the model’s answer
with torch.no_grad():
    output_tokens = llm_model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and clean the output
answer = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

# Extract the part after "Answer:" if the model echoes the prompt
if "Answer:" in answer:
    answer = answer.split("Answer:")[-1].strip()

print("💬 RAG Answer:\n")
print(answer)


💬 RAG Answer:

The 'LLM in a Flash' paper proposes a method to speed up inference by selectively loading parameters on demand for each token generation step. The novelty of this approach is that it can achieve a 3x speed up over the naive baseline (Table 3) while performing better than the hybrid model, which is the theoretical lower bound for approaches that don't use sparsity. The method was applied to the Phi-2 model, and the inference speed-up was achieved by modifying the window size to ensure it never exceeds the limit. The results show that the approach can be effective even when the model is already small, as demonstrated by the application to the Phi-2 model.


# Load LLaVA Model

In [4]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig
)

model_id = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # efficient for T4
)

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✅ LLaVA loaded successfully on T4 in 4-bit mode.")


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

✅ LLaVA loaded successfully on T4 in 4-bit mode.


In [17]:
question = "Tell me about the 'LLM in a Flash' paper. What is the novelty, what methods they proposed, and what inference speed up they got?"

text_hits, image_hits = retrieve_multimodal(question, k_text=5, k_image=3)

# Build text context from top hits
text_context = "\n\n".join([f"[Text: page {t['page']}] {t['content_preview']}" for t in text_hits])

# Collect image paths
image_paths = [h["image_path"] for h in image_hits]


In [18]:
prompt = f"""
You are an expert AI researcher summarizing the paper based only on the retrieved content.
Integrate textual and visual information logically.
If an image is included, describe what it shows and how it relates to the text.
Keep your answer academic and concise (1–2 paragraphs).

Question: {question}

Context:
{text_context}

Answer:
"""


In [19]:
from PIL import Image

# Load all retrieved images
images = [Image.open(path).convert("RGB") for path in image_paths]


In [20]:
inputs = processor(
    text=prompt,
    images=images,   # pass list of PIL images
    return_tensors="pt"
).to("cuda", torch.float16)


In [ ]:
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

answer = processor.decode(output[0], skip_special_tokens=True)
print("🧠 RAG + LLaVA Answer:\n")
print(answer)


In [24]:
test_prompt = "What do you see in this image? <image>"
inputs = processor(text=test_prompt, images=[images[0]], return_tensors="pt").to("cuda", torch.float16)
output = model.generate(**inputs, max_new_tokens=100)
print(processor.decode(output[0], skip_special_tokens=True))


What do you see in this image? 

The image shows a comparison of different types of computers. There are four different types of computers, each with its own specifications. The first computer is a LIMA 2-7B, which is a 2.7 GHz processor. The second computer is a LIMA 2-7B, which is a 2.7 GHz processor. The third computer is a LIMA 2-7B, which is a 2.7 GHz processor.


In [25]:
print(image_paths[0])


data/rag_assets/images/2312.11514/figure/figure_1.png


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig

model_id = "microsoft/phi-3-vision-128k-instruct"

# 4-bit quantization for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Phi-3-Vision loaded successfully on T4 in 4-bit mode.")
